# Pipeline End-to-End de Deep Learning con Fashion-MNIST

**Clasificación de imágenes con TensorFlow/Keras desde los datos hasta la predicción**

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luisf2020/Proyectos-de-Inteligencia-Artificial/blob/main/notebooks/05_end_to_end_fashion_mnist.ipynb)

> Implementación desarrollada específicamente para este repositorio como proyecto técnico de portafolio.

## Objetivo

Construir un flujo **end-to-end**: cargar imágenes, normalizar píxeles, definir una red neuronal, entrenarla, evaluar su rendimiento y visualizar predicciones. El modelo aprende directamente desde las intensidades de los píxeles hasta una de diez categorías de prendas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = [
    "Camiseta/top", "Pantalón", "Suéter", "Vestido", "Abrigo",
    "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"
]

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for ax, image, label in zip(axes.ravel(), X_train[:10], y_train[:10]):
    ax.imshow(image, cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.20),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=8,
    batch_size=128,
    verbose=1
)

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Loss de prueba: {test_loss:.4f}")
print(f"Accuracy de prueba: {test_accuracy:.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Entrenamiento")
plt.plot(history.history["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Evolución del aprendizaje")
plt.legend()
plt.show()

In [ ]:
probabilities = model.predict(X_test[:12], verbose=0)
predictions = probabilities.argmax(axis=1)

fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax, image, real, predicted, confidence in zip(
    axes.ravel(),
    X_test[:12],
    y_test[:12],
    predictions,
    probabilities.max(axis=1)
):
    ax.imshow(image, cmap="gray")
    ax.set_title(
        f"Real: {class_names[real]}\nPred: {class_names[predicted]} ({confidence:.1%})"
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

## Conclusiones

Un flujo end-to-end reduce la necesidad de diseñar manualmente características: la red aprende representaciones útiles desde los datos de entrada. Aun así, una red densa no aprovecha explícitamente la estructura espacial de las imágenes; una CNN sería un siguiente paso natural.